<a href="https://colab.research.google.com/github/shreyaakamra/pdf_q-a/blob/main/notebooks/pdf_qa_day5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Step 1: Install all dependencies
!pip install -q pypdf langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu transformers gradio accelerate torch

# Step 2: Clone your repo fresh from GitHub
!git clone https://github.com/shreyaakamra/pdf_q-a.git /content/pdf-qa

# Step 3: Verify structure
import os
print(os.listdir('/content/pdf-qa'))
print(os.listdir('/content/pdf-qa/src'))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Cloning into '/content/pdf-qa'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 34 (delta 9), reused 9 (delta 0), pack-reused 0 (from 0)
Receiving objects: 

In [4]:
# Create it locally
with open('/content/pdf-qa/src/__init__.py', 'w') as f:
    pass

print("src contents now:", os.listdir('/content/pdf-qa/src'))

src contents now: ['llm.py', 'config.py', '__init__.py', '__pycache__', 'pdf_processing.py']


In [6]:
%cd /content/pdf-qa
!git config --global user.email "shreyaakamra@gmail.com"
!git config --global user.name "ShreyaaKamra"

from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git remote set-url origin https://{token}@github.com/shreyaakamra/pdf_q-a.git
!git add src/__init__.py
!git commit -m "Fix: add missing src/__init__.py"
!git push origin main

/content/pdf-qa
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


In [2]:
import sys
sys.path.insert(0, '/content/pdf-qa')

from src.pdf_processing import process_pdf
from src.llm import answer_question, build_prompt, generate_text
from src import config
print("All imports OK")
print("LLM:", config.LLM_MODEL)

All imports OK
LLM: google/flan-t5-large


In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_community.embeddings import HuggingFaceEmbeddings

print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODEL)

print("Loading language model...")
tokenizer = AutoTokenizer.from_pretrained(config.LLM_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(config.LLM_MODEL)

print("All models loaded and ready.")

Loading embedding model...


/tmp/ipykernel_1690/2147990937.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODEL)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

All models loaded and ready.


In [7]:
import gradio as gr

state = {"vectorstore": None}

def upload_and_process(file):
    if file is None:
        return "No file uploaded."
    try:
        vectorstore = process_pdf(file.name, embeddings)
        state["vectorstore"] = vectorstore
        return " PDF processed successfully. You can now ask questions below."
    except ValueError as e:
        return f"⚠️ Error: {str(e)}"
    except Exception as e:
        return f" Unexpected error: {str(e)}"

def ask_question(question):
    if not question.strip():
        return "Please enter a question.", ""
    if state["vectorstore"] is None:
        return "Please upload a PDF first.", ""
    try:
        answer, source_chunks = answer_question(
            state["vectorstore"], question, tokenizer, model, k=config.RETRIEVAL_K
        )
        sources = "\n\n---\n\n".join([
            f"Source chunk {i+1}:\n{chunk.page_content[:300]}..."
            for i, chunk in enumerate(source_chunks[:3])
        ])
        return answer, sources
    except Exception as e:
        return f"Error: {str(e)}", ""

with gr.Blocks(title="PDF Q&A System") as demo:
    gr.Markdown("""
    # 📄 PDF Q&A System
    Upload any PDF and ask questions about its content.
    Answers are grounded in the document — the system will say
    **"I don't know"** if your question isn't covered.
    """)

    with gr.Row():
        with gr.Column():
            pdf_input = gr.File(
                label="Upload PDF",
                file_types=[".pdf"]
            )
            upload_status = gr.Textbox(
                label="Upload Status",
                interactive=False
            )

    with gr.Row():
        with gr.Column():
            question_input = gr.Textbox(
                label="Ask a question about the document",
                placeholder="e.g. What is the Windows API?",
                lines=2
            )
            ask_button = gr.Button("Get Answer", variant="primary")

    with gr.Row():
        with gr.Column():
            answer_output = gr.Textbox(
                label="Answer",
                interactive=False,
                lines=4
            )
            sources_output = gr.Textbox(
                label="Source chunks used to generate this answer",
                interactive=False,
                lines=6
            )

    pdf_input.change(
        fn=upload_and_process,
        inputs=pdf_input,
        outputs=upload_status
    )
    ask_button.click(
        fn=ask_question,
        inputs=question_input,
        outputs=[answer_output, sources_output]
    )
    question_input.submit(
        fn=ask_question,
        inputs=question_input,
        outputs=[answer_output, sources_output]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://05d4ac10f99442a745.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
